In [1]:
import pandas as pd
import json
import os
import gzip
import numpy as np

In [2]:
from pathlib import Path

# Robustly determine project root (search for folder name or marker files)
def find_project_root(start=Path.cwd(), target_name="RAG_LUMC"):
    p = start.resolve()
    for parent in [p] + list(p.parents):
        if parent.name == target_name:
            return parent
    # Fallback: search for marker files like run_all_files.py or README.md
    for parent in [p] + list(p.parents):
        if (parent / "run_all_files.py").exists() or (parent / "README.md").exists():
            return parent
    return start.resolve()

PROJECT_ROOT = find_project_root()
# GSEA directory
GSEA_DIR = PROJECT_ROOT / "data" / "GSEA"

print("Project root:", PROJECT_ROOT)
print("GSEA dir:", GSEA_DIR)
print("Exists:", GSEA_DIR.exists())


Project root: C:\Users\misha\RAG_LUMC
GSEA dir: C:\Users\misha\RAG_LUMC\data\GSEA
Exists: True


In [3]:
# =========================
# Central configuration (edit only this cell for new datasets/species)
# =========================
SPECIES = "Mus_musculus"
SPECIES_SHORT = "mice"
WIKIPATHWAYS_DATE = "20260210"

RAW_BIOMART_FILE = f"{SPECIES_SHORT}_data.txt.gz"
REORDERED_BIOMART_FILE = f"reordered_{RAW_BIOMART_FILE}"
CONSOLIDATED_GENES_FILE = GSEA_DIR / "external_gene_data" / f"{SPECIES_SHORT}_genes_consolidated.txt.gz"
GENES_JSON_PATH = PROJECT_ROOT / "data" / "JSON" / "genes.json"

WIKIPATHWAYS_INPUT_FILE = GSEA_DIR / "to_be_converted" / f"wikipathways-{WIKIPATHWAYS_DATE}-gmt-{SPECIES}.gmt"
WIKIPATHWAYS_CONVERTED_FILE = GSEA_DIR / "to_be_converted" / f"converted_wikipathways-{WIKIPATHWAYS_DATE}-gmt-{SPECIES}.gmt"
WIKIPATHWAYS_SYNONYMS_OUTPUT_FILE = GSEA_DIR / "external_gene_data" / f"wikipathways_synonyms_{SPECIES}.gmt.gz"

MAX_GENES = 250
EXCEL_INPUT_FILE = GSEA_DIR / "genes_of_interest" / "C3_9w_detab_fixed.xlsx"
GENE_DESCRIPTIONS_OUTPUT_CSV = GSEA_DIR / "external_gene_data" / "gene_descriptions.csv"

print("Configured species:", SPECIES)
print("Raw Biomart file:", GSEA_DIR / "to_be_converted" / RAW_BIOMART_FILE)
print("WikiPathways input:", WIKIPATHWAYS_INPUT_FILE)
print("Consolidated genes output:", CONSOLIDATED_GENES_FILE)

Configured species: Mus_musculus
Raw Biomart file: C:\Users\misha\RAG_LUMC\data\GSEA\to_be_converted\mice_data.txt.gz
WikiPathways input: C:\Users\misha\RAG_LUMC\data\GSEA\to_be_converted\wikipathways-20260210-gmt-Mus_musculus.gmt
Consolidated genes output: C:\Users\misha\RAG_LUMC\data\GSEA\external_gene_data\mice_genes_consolidated.txt.gz


In [4]:
input_path = GSEA_DIR / "to_be_converted" / RAW_BIOMART_FILE
output_path = GSEA_DIR / "to_be_converted" / REORDERED_BIOMART_FILE

# Ensure the input file exists
if not os.path.exists(input_path):
    print(f"Error: File '{input_path}' does not exist.")
else:
    # Define the columns to extract
    cols = [
        "Gene stable ID",
        "Gene name",
        "Gene description",
        "Gene Synonym",
        "NCBI gene (formerly Entrezgene) description",
        "NCBI gene (formerly Entrezgene) ID"
    ]
    
    try:
        # Read the gzipped CSV file
        df = pd.read_csv(input_path, compression="gzip")
    except Exception as e:
        print("Error reading input file:", e)
    else:
        # Check if all required columns exist
        missing_cols = [col for col in cols if col not in df.columns]
        if missing_cols:
            print("Error: The following required columns are missing:", missing_cols)
        else:
            # Select and reorder the desired columns
            df_selected = df[cols]
            
            try:
                # Write the output to a new gzipped CSV file
                df_selected.to_csv(output_path, index=False, compression="gzip")
                print(f"Reordered file saved as {output_path}")
            except Exception as e:
                print("Error writing output file:", e)


Error reading input file: Error tokenizing data. C error: Expected 1 fields in line 130, saw 2



In [5]:
# Not changing working directory in the notebook; project root detection handles paths

In [6]:
print(os.getcwd())

c:\Users\misha\RAG_LUMC\supporting scripts


In [7]:
Add_Synonyms = True

In [8]:
# Bestaande project paden
JSON_DIR = PROJECT_ROOT / "data" / "JSON"

# Input/output bestanden
input_file = WIKIPATHWAYS_INPUT_FILE
output_file = WIKIPATHWAYS_CONVERTED_FILE
json_file = JSON_DIR / "ncbi_id_to_symbol.json"

# Load the gene ID to symbol dictionary from JSON
with open(json_file, 'r') as f:
    gene_dict = json.load(f)

# Read the GMT input file
columns = ['header', 'url'] + [f'gene_{i}' for i in range(1000)]  
df = pd.read_csv(input_file, sep='\t', header=None, names=columns, engine='python', dtype=str, na_filter=False)

# Function to replace gene IDs with symbols
def replace_gene_ids(gene_id):
    return gene_dict.get(gene_id, gene_id)

# Apply replacement function to gene columns
gene_columns = df.columns[2:]
for col in gene_columns:
    df[col] = df[col].apply(replace_gene_ids)

# Remove empty columns (if all values in a column are empty strings)
df = df.loc[:, (df != '').any(axis=0)]

# Remove empty rows for gene columns to prevent excess line breaks
df = df.apply(lambda x: x.dropna().tolist(), axis=1).apply(pd.Series)

# Save to the output file without excess newlines
df.to_csv(output_file, sep='\t', header=False, index=False, lineterminator='\n')

print(f'File conversion completed! Output saved to {output_file}')


File conversion completed! Output saved to C:\Users\misha\RAG_LUMC\data\GSEA\to_be_converted\converted_wikipathways-20260210-gmt-Mus_musculus.gmt


In [9]:
# Configuration preview
print("Reordered Biomart input:", GSEA_DIR / "to_be_converted" / REORDERED_BIOMART_FILE)
print("Genes JSON output:", GENES_JSON_PATH)
print("Consolidated genes output:", CONSOLIDATED_GENES_FILE)

Reordered Biomart input: C:\Users\misha\RAG_LUMC\data\GSEA\to_be_converted\reordered_mice_data.txt.gz
Genes JSON output: C:\Users\misha\RAG_LUMC\data\JSON\genes.json
Consolidated genes output: C:\Users\misha\RAG_LUMC\data\GSEA\external_gene_data\mice_genes_consolidated.txt.gz


In [10]:
from pathlib import Path
import os
import json
import pandas as pd

# Absolute paden gebruiken
input_file = GSEA_DIR / "to_be_converted" / REORDERED_BIOMART_FILE
raw_input_file = GSEA_DIR / "to_be_converted" / RAW_BIOMART_FILE
output_json = GENES_JSON_PATH
output_file = CONSOLIDATED_GENES_FILE

# Functie om directory te maken
def ensure_dir(file_path: Path):
    directory = file_path.parent
    if not directory.exists():
        directory.mkdir(parents=True, exist_ok=True)
        print(f"Created directory: {directory}")

def read_biomart_with_fallbacks(raw_path: Path):
    """Read biomart export with delimiter fallbacks."""
    read_attempts = [
        {"kwargs": {"compression": "gzip"}, "label": "comma (default)"},
        {"kwargs": {"compression": "gzip", "sep": "\t"}, "label": "tab"},
        {"kwargs": {"compression": "gzip", "sep": ";"}, "label": "semicolon"},
        {"kwargs": {"compression": "gzip", "sep": None, "engine": "python"}, "label": "auto-detect"},
    ]

    last_error = None
    for attempt in read_attempts:
        try:
            df_tmp = pd.read_csv(raw_path, **attempt["kwargs"] )
            print(f"Read raw biomart using delimiter mode: {attempt['label']}")
            return df_tmp
        except Exception as err:
            last_error = err

    raise ValueError(
        f"Could not parse raw biomart file: {raw_path}. Last error: {last_error}"
    )

def build_reordered_biomart_if_needed(reordered_path: Path, raw_path: Path):
    """Create reordered biomart file if missing."""
    if reordered_path.exists():
        print(f"Using existing reordered biomart file: {reordered_path}")
        return

    if not raw_path.exists():
        available = sorted([p.name for p in raw_path.parent.glob("*.gz")])
        raise FileNotFoundError(
            f"Neither reordered nor raw biomart input exists.\n"
            f"Expected reordered: {reordered_path}\n"
            f"Expected raw: {raw_path}\n"
            f"Available .gz files in {raw_path.parent}: {available}"
        )

    cols = [
        "Gene stable ID",
        "Gene name",
        "Gene description",
        "Gene Synonym",
        "NCBI gene (formerly Entrezgene) description",
        "NCBI gene (formerly Entrezgene) ID"
    ]

    raw_df = read_biomart_with_fallbacks(raw_path)
    missing_cols = [col for col in cols if col not in raw_df.columns]
    if missing_cols:
        raise ValueError(
            f"Raw biomart file is missing required columns: {missing_cols}. "
            f"Available columns: {list(raw_df.columns)}"
        )

    raw_df[cols].to_csv(reordered_path, index=False, compression="gzip")
    print(f"Created reordered biomart file: {reordered_path}")

# Maak benodigde mappen aan
ensure_dir(output_file)
ensure_dir(output_json)
ensure_dir(input_file)

# Zorg dat de reordered input bestaat
build_reordered_biomart_if_needed(input_file, raw_input_file)

# Lees input
df = pd.read_csv(input_file, compression='gzip')
print(f"Successfully read input file: {input_file}")

df['Gene Synonym'] = df['Gene Synonym'].fillna('').astype(str).str.strip()

df['Gene stable ID'] = df['Gene stable ID'].str.strip().str.upper()

unique_gene_ids = df['Gene stable ID'].nunique()
print(f"Number of unique 'Gene stable ID's: {unique_gene_ids}")
# Grouping the data and applying transformations
grouped = df.groupby('Gene stable ID').agg({
    'Gene name': 'first',
    'Gene description': 'first',
    'Gene Synonym': lambda x: sorted(filter(None, x.unique())),
    'NCBI gene (formerly Entrezgene) description': 'first',
    'NCBI gene (formerly Entrezgene) ID': 'first'
}).reset_index()
# Ensure 'Gene Synonyms' are formatted correctly
grouped.rename(columns={'Gene Synonym': 'Gene Synonyms'}, inplace=True)
grouped['Gene Synonyms'] = grouped['Gene Synonyms'].apply(lambda x: f"[{','.join(x)}]" if x else "[]")

# 1. Removing NCBI description unless Gene description is empty and NCBI description is not
grouped['Gene description'] = grouped.apply(
    lambda row: row['NCBI gene (formerly Entrezgene) description']
    if pd.isna(row['Gene description']) and pd.notna(row['NCBI gene (formerly Entrezgene) description'])
    else row['Gene description'],
    axis=1
)

# Removing descriptions between [].
grouped['Gene description'] = grouped['Gene description'].str.replace(r'\[.*?\]', '', regex=True).str.strip()

# Dropping the NCBI description column as it's no longer needed, and converting the ID to integer.
grouped.drop(columns=['NCBI gene (formerly Entrezgene) description'], inplace=True)
grouped['NCBI gene (formerly Entrezgene) ID'] = grouped['NCBI gene (formerly Entrezgene) ID'].astype('Int64')

consolidated_entries = grouped.shape[0]
print(f"Number of consolidated entries: {consolidated_entries}")

if consolidated_entries != unique_gene_ids:
    print("Warning: The number of consolidated entries does not match the number of unique 'Gene stable ID's.")
    print(f"Unique 'Gene stable ID's: {unique_gene_ids}, Consolidated entries: {consolidated_entries}")
else:
    print("Success: The number of consolidated entries matches the number of unique 'Gene stable ID's.")

genes_list = grouped.to_dict(orient='records')

with open(output_json, 'w', encoding='utf-8') as f_json:
    json.dump(genes_list, f_json, indent=4)
print(f"Consolidated JSON data has been saved to '{output_json}'.")

ordered_columns = [
    'Gene stable ID',
    'Gene name',
    'Gene description',
]

missing_columns = set(ordered_columns) - set(grouped.columns)
if missing_columns:
    raise ValueError(f"Missing columns in the DataFrame: {missing_columns}")

grouped_ordered = grouped[ordered_columns]

grouped_ordered.to_csv(output_file, index=False, sep=',', compression='gzip')
print(f"Consolidated TXT.GZ data has been saved to '{output_file}'.")

Read raw biomart using delimiter mode: tab
Created reordered biomart file: C:\Users\misha\RAG_LUMC\data\GSEA\to_be_converted\reordered_mice_data.txt.gz
Successfully read input file: C:\Users\misha\RAG_LUMC\data\GSEA\to_be_converted\reordered_mice_data.txt.gz
Number of unique 'Gene stable ID's: 78334
Number of consolidated entries: 78334
Success: The number of consolidated entries matches the number of unique 'Gene stable ID's.
Consolidated JSON data has been saved to 'C:\Users\misha\RAG_LUMC\data\JSON\genes.json'.
Consolidated TXT.GZ data has been saved to 'C:\Users\misha\RAG_LUMC\data\GSEA\external_gene_data\mice_genes_consolidated.txt.gz'.


In [11]:
# Configuration preview
print("Genes JSON input:", GENES_JSON_PATH)
print("Converted GMT input:", WIKIPATHWAYS_CONVERTED_FILE)
print("Synonyms GMT output:", WIKIPATHWAYS_SYNONYMS_OUTPUT_FILE)

Genes JSON input: C:\Users\misha\RAG_LUMC\data\JSON\genes.json
Converted GMT input: C:\Users\misha\RAG_LUMC\data\GSEA\to_be_converted\converted_wikipathways-20260210-gmt-Mus_musculus.gmt
Synonyms GMT output: C:\Users\misha\RAG_LUMC\data\GSEA\external_gene_data\wikipathways_synonyms_Mus_musculus.gmt.gz


In [12]:
from pathlib import Path
import gzip
import json

# =========================
# Input / output paden (from central config)
# =========================
genes_json_path = GENES_JSON_PATH
input_gmt_path = WIKIPATHWAYS_CONVERTED_FILE
output_gmt_path = WIKIPATHWAYS_SYNONYMS_OUTPUT_FILE

# =========================
# Safety checks
# =========================
for path in [genes_json_path, input_gmt_path]:
    if not path.exists():
        raise FileNotFoundError(f"Input file not found: {path}")

# Ensure output directory exists
output_gmt_path.parent.mkdir(parents=True, exist_ok=True)

# =========================
# Load genes data
# =========================
with open(genes_json_path, "r", encoding="utf-8") as f_json:
    genes_data = json.load(f_json)

# Create gene → synonyms mapping
gene_to_synonyms = {}

for entry in genes_data:
    gene_name = entry.get("Gene name")
    if not gene_name:
        continue

    gene_name = gene_name.strip()

    synonyms_str = (entry.get("Gene Synonyms") or "").strip()
    if synonyms_str.startswith("[") and synonyms_str.endswith("]"):
        synonyms_str = synonyms_str[1:-1]

    synonyms = [s.strip() for s in synonyms_str.split(",") if s.strip()]
    gene_to_synonyms[gene_name] = synonyms

print(f"Loaded {len(gene_to_synonyms)} genes with synonyms.")

# =========================
# Process GMT file
# =========================
def process_gmt(input_path: Path, output_path: Path, gene_synonyms_map):
    base_url = "https://www.wikipathways.org/instance/"

    with open(input_path, "r", encoding="utf-8") as infile, \
         gzip.open(output_path, "wt", encoding="utf-8") as outfile:

        for line_number, line in enumerate(infile, 1):
            line = line.strip()
            if not line:
                continue

            parts = line.split("\t")
            if len(parts) < 3:
                print(
                    f"Warning: Line {line_number} does not have enough columns. Skipping."
                )
                continue

            pathway_name_full, pathway_url_full, *genes = parts

            pathway_name = pathway_name_full.split("%")[0].strip()
            pathway_url = (
                pathway_url_full.replace(base_url, "").strip()
                if pathway_url_full.startswith(base_url)
                else pathway_url_full.strip()
            )

            expanded_genes = []
            for gene in genes:
                gene = gene.strip()
                if not gene:
                    continue

                synonyms = gene_synonyms_map.get(gene, [])
                expanded_genes.append(
                    f"[{gene}, {', '.join(synonyms)}]" if synonyms else f"[{gene}]"
                )

            # Remove duplicates (order preserved)
            seen = set()
            unique_genes = []
            for g in expanded_genes:
                if g not in seen:
                    seen.add(g)
                    unique_genes.append(g)

            outfile.write(
                "\t".join([pathway_name, pathway_url] + unique_genes) + "\n"
            )

            if line_number % 1000 == 0:
                print(f"Processed {line_number} lines.")

    print(f"Finished processing GMT file. Output saved to {output_path}")

# =========================
# Run
# =========================
process_gmt(input_gmt_path, output_gmt_path, gene_to_synonyms)


Loaded 77495 genes with synonyms.
Finished processing GMT file. Output saved to C:\Users\misha\RAG_LUMC\data\GSEA\external_gene_data\wikipathways_synonyms_Mus_musculus.gmt.gz


In [13]:
# Import necessary libraries
import pandas as pd
import gzip
import csv
from pathlib import Path

# Input/output bestanden (from central config)
excel_file_path = EXCEL_INPUT_FILE
gene_data_file = CONSOLIDATED_GENES_FILE
output_csv = GENE_DESCRIPTIONS_OUTPUT_CSV

# =========================
# Safety check
# =========================
for path in [excel_file_path, gene_data_file]:
    if not path.exists():
        raise FileNotFoundError(f"Input file not found: {path}")

# =========================
# Functies
# =========================
def initialize_gene_list(
    excel_file_path: Path,
    de_filter_option="combined",
    test=False
):
    """
    Initializes the gene list by processing the Excel file.
    """
    results = process_excel_data(excel_file_path, de_filter_option, test)
    if results:
        gene_list_string, regulation, num_genes = results[0]
    else:
        gene_list_string = ""
        regulation = ""
        num_genes = 0

    return gene_list_string, regulation, num_genes


def process_excel_data(
    excel_file_path: Path,
    de_filter_option,
    test
):
    """
    Processes the Excel data to filter genes based on DE and FDR thresholds.
    """
    data = pd.read_excel(excel_file_path)
    results = []
    fdr_threshold = 0.00008802967327

    if not test:
        if de_filter_option == "combined":
            data = data[data['DE'] != 0]
            data = data[data['FDR'] <= fdr_threshold]

            if not data.empty:
                genes_list = data['X'].tolist()
                num_genes = len(genes_list)
                unique_de_values = data['DE'].unique()

                regulation = (
                    "upregulated" if len(unique_de_values) == 1 and unique_de_values[0] == 1 else
                    "downregulated" if len(unique_de_values) == 1 and unique_de_values[0] == -1 else
                    "combined"
                )
                results.append((', '.join(genes_list), regulation, num_genes))

        elif de_filter_option == "separate":
            for de_value, regulation_label in [(1, "upregulated"), (-1, "downregulated")]:
                filtered_data = data[data['DE'] == de_value]
                filtered_data = filtered_data[filtered_data['FDR'] <= fdr_threshold]

                if not filtered_data.empty:
                    genes_list = filtered_data['X'].tolist()
                    num_genes = len(genes_list)
                    results.append((', '.join(genes_list), regulation_label, num_genes))
        else:
            raise ValueError("Invalid DE filter option. Use 'combined' or 'separate'.")

    return results


def extract_gene_descriptions(
    gene_list_string,
    gene_data_file: Path,
    output_csv: Path
):
    if not gene_list_string:
        print("Gene list is empty. No descriptions to extract.")
        return

    # Parse gene list
    gene_names = [g.strip() for g in gene_list_string.split(',') if g.strip()]
    gene_names_set = set(gene_names)

    print(f"Total genes to process: {len(gene_names_set)}")

    gene_description_dict = {}

    with gzip.open(gene_data_file, 'rt', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            gene_name = row.get('Gene name', '').strip()
            if gene_name in gene_names_set:
                gene_description_dict[gene_name] = row.get(
                    'Gene description', ''
                ).strip()
                gene_names_set.remove(gene_name)
                if not gene_names_set:
                    break

    output_df = pd.DataFrame({
        "Gene name": gene_names,
        "Gene description": [
            gene_description_dict.get(g, "Description not found")
            for g in gene_names
        ]
    })

    output_df.to_csv(output_csv, index=False)
    print(f"Successfully created '{output_csv}' with {len(output_df)} entries.")

    if gene_names_set:
        print(
            "The following genes were not found in the gene data file:",
            ", ".join(sorted(gene_names_set))
        )

# =========================
# Run pipeline
# =========================
gene_list_string, regulation, num_genes = initialize_gene_list(
    excel_file_path=excel_file_path,
    de_filter_option="combined",
    test=False
)

extract_gene_descriptions(
    gene_list_string=gene_list_string,
    gene_data_file=gene_data_file,
    output_csv=output_csv
)


Total genes to process: 505
Successfully created 'C:\Users\misha\RAG_LUMC\data\GSEA\external_gene_data\gene_descriptions.csv' with 505 entries.
The following genes were not found in the gene data file: Dnaic2, Fam57a, Gm21887, Gm5936, Ick, Mpp6, Olfr1033, Pcnx, Pnmal2, Sept9, Tmem28, Vdac3.ps1, X2310014F06Rik, X2310022B05Rik, X4632428C04Rik, X4931428F04Rik, X9430041J12Rik
